In [ ]:
# Environment Setup
import os 
import sys

REPO_NAME = "A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring"
REPO_URL = "https://github.com/SnowWoofer/A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring.git" # could make .env or dynamic

if os.path.exists("/kaggle"):
    BASE_DIR = f"/kaggle/working/{REPO_NAME}"
    os.environ["DATA_DIR"] = f"/kaggle/working/{REPO_NAME}"
elif os.path.exists("/content"):    
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = "/content/drive/MyDrive/blockchain_misinformation_data"
    os.makedirs(os.environ["DATA_DIR"], exist_ok=True)
    BASE_DIR = f"/content/{REPO_NAME}"
else:
    BASE_DIR = "."

if not os.path.exists(BASE_DIR):
    !git clone {REPO_URL} {REPO_NAME}

os.chdir(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "src"))

In [ ]:
# Fix requirements.txt encoding if it's UTF-16 (Windows pip freeze), then install
req_path = "requirements.txt"
with open(req_path, "rb") as f:
    raw = f.read()
if raw.startswith(b"\xff\xfe") or raw.startswith(b"\xfe\xff"):
    text = raw.decode("utf-16")
    with open(req_path, "w", encoding="utf-8") as f:
        f.write(text)

!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Check Settings > Accelerator > GPU in the Kaggle sidebar.")

In [ ]:
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

seed = 100
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
df = pd.read_csv("/kaggle/input/your-dataset-name/twitter_data_eng_raw.csv", encoding="utf-8")
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

train_df = df[df["set"] == "train"][["text", "label"]].reset_index(drop=True)
test_df = df[df["set"] == "test"][["text", "label"]].reset_index(drop=True)

train_combined = pd.concat([train_df_pt1,train_df_pt2], )

print(f"train: {len(train_df)} rows, test: {len(test_df)} rows")
print(train_df["label"].value_counts())

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
model_path = "Davlan/afro-xlmr-large"

id2label = {0: "not misinformation", 1: "misinformation"}
label2id = {"not misinformation": 0, "misinformation": 1}

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# Freeze all base model params except the pooling layer (near the classification head)
for name, param in model.base_model.named_parameters():
    param.requires_grad = "pooler" in name

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [ ]:
sanity_train = tokenized_train.select(range(min(100, len(tokenized_train))))
sanity_test = tokenized_test.select(range(min(50, len(tokenized_test))))

sanity_args = TrainingArguments(
    output_dir="/kaggle/working/sanity_check",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=5,
    report_to="none",
)

sanity_trainer = Trainer(
    model=model,
    args=sanity_args,
    train_dataset=sanity_train,
    eval_dataset=sanity_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

sanity_trainer.train()

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/afroxlmr_eng_baseline",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

trainer.save_model("/kaggle/working/afroxlmr_eng_baseline_final")
tokenizer.save_pretrained("/kaggle/working/afroxlmr_eng_baseline_final")